In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import lib_debug

%load_ext autoreload
%autoreload 2


# Robust multi-objective Bayesian optimization

This notebook is based on `gaussian_process_debug_robust.ipynb`. The scalar target uses the multi-objective synthetic function from `synthetic_mo_robust.ipynb` and optimizes `robust_weighted_average`.


In [ ]:
# -------------------- Robust multi-objective synthetic function --------------------
# robust_pareto_ridge is taken from synthetic_mo_robust.ipynb.
# robust_weighted_average is the scalar minimization target used by the BO loop.


def robust_pareto_ridge(
    X,
    a=(0.20, 0.20),
    b=(0.72, 0.72),
    sigma_z=0.35,
    sigma_min=0.015,
    sigma_max=0.120,
    tau=0.32,
):
    """
    Robust multi-objective synthetic function.

    Minimization problem:
        minimize (f1(x), f2(x))

    Internally:
        f_j(x) = -g_j(x)

    Shape:
        - nominal Pareto set is approximately the line segment from a to b
        - endpoints are nominally good but fragile
        - central region is less sharp and more robust to perturbations
    """
    X = np.asarray(X, dtype=float)

    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    v = b - a
    vv = np.dot(v, v)

    # coordinate along the Pareto line
    z = np.sum((X - a) * v, axis=-1) / vv

    # projection onto the infinite line through a and b
    pi = a + z[..., None] * v

    # squared perpendicular distance
    r2 = np.sum((X - pi) ** 2, axis=-1)

    # z-dependent perpendicular width
    sigma_perp = sigma_min + (sigma_max - sigma_min) * np.exp(
        -((z - 0.5) / tau) ** 4
    )

    # common perpendicular factor
    perp_factor = np.exp(-r2 / (2.0 * sigma_perp**2))

    # objective-specific trade-off along z
    g1 = np.exp(-(z**2) / (2.0 * sigma_z**2)) * perp_factor
    g2 = np.exp(-((1.0 - z) ** 2) / (2.0 * sigma_z**2)) * perp_factor

    f1 = -g1
    f2 = -g2

    return np.stack([f1, f2], axis=-1)


def robust_average_objectives(
    X,
    func=robust_pareto_ridge,
    n_delta=512,
    delta_std=0.04,
    rng=None,
    clip=True,
):
    """Monte Carlo robust average of each objective under Gaussian input perturbations."""
    rng = np.random.default_rng() if rng is None else rng
    X = np.asarray(X, dtype=float)
    if X.shape[-1] != 2:
        raise ValueError("robust_pareto_ridge expects the last input dimension to be 2.")

    deltas = rng.normal(0.0, delta_std, size=(n_delta, X.shape[-1]))
    X_perturbed = X[..., None, :] + deltas
    if clip:
        X_perturbed = np.clip(X_perturbed, 0.0, 1.0)

    F_perturbed = func(X_perturbed)
    return np.mean(F_perturbed, axis=-2)


def robust_weighted_average(
    X,
    weights=(0.5, 0.5),
    func=robust_pareto_ridge,
    n_delta=512,
    delta_std=0.04,
    rng=None,
    clip=True,
):
    """
    Robust scalar objective for minimization.

    Computes:
        S_w(x) = w1 * bar_f1(x) + w2 * bar_f2(x)

    Since f1 and f2 are negative-valued objectives,
    smaller values are better.
    """

    weights = np.asarray(weights, dtype=float)
    weights = weights / weights.sum()

    F_bar = robust_average_objectives(
        X=X,
        func=func,
        n_delta=n_delta,
        delta_std=delta_std,
        rng=rng,
        clip=clip,
    )

    return F_bar @ weights


In [ ]:
# -------------------- Synthetic function 3D projection plots --------------------
# The synthetic functions are implemented as minimization objectives in lib_debug.py.
# Switch the function and scatter points by comment-in / comment-out.


def _as_2d_points(points):
    """Return points as an (n, 2) array, or None when no points are provided."""
    if points is None:
        return None
    points = np.asarray(points, dtype=float)
    if points.size == 0:
        return None
    return points.reshape(-1, 2)


def plot_synthetic_function_3d(
    function,
    n_grid=121,
    initial_points=None,
    exploration_points=None,
    x_initial=None,
    y_initial=None,
):
    """Plot a 3D surface and a 2D projection with an optional BO search trace.

    initial_points can be an (n, 2) array. For a single initial coordinate, pass
    x_initial and y_initial and it will be connected with the exploration points.
    """
    x1 = np.linspace(0.0, 1.0, n_grid)
    x2 = np.linspace(0.0, 1.0, n_grid)
    X1, X2 = np.meshgrid(x1, x2)
    X = np.stack([X1, X2], axis=-1)
    Z = function(X)

    z_min = float(np.min(Z))
    z_max = float(np.max(Z))
    fig = plt.figure(figsize=(7, 5))
    ax = fig.add_subplot(111, projection="3d")
    surface = ax.plot_surface(X1, X2, Z, cmap="viridis", linewidth=0, antialiased=True, alpha=0.88)
    ax.contour(X1, X2, Z, zdir="z", offset=z_min, cmap="viridis", levels=20)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_zlabel("f(x)")
    ax.set_zlim(z_min, z_max)
    fig.colorbar(surface, ax=ax, shrink=0.65, pad=0.1)
    plt.tight_layout()

    fig, ax = plt.subplots(1, 1, figsize=(8, 6), dpi=80)
    cont = ax.contourf(X1, X2, Z, levels=40, cmap="viridis")
    ax.contour(X1, X2, Z, levels=12, colors="white", linewidths=0.5, alpha=0.6)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_aspect("equal", adjustable="box")
    fig.colorbar(cont, ax=ax)
    ax.axhline(0, color="white", linestyle="--", linewidth=1.0)
    ax.axvline(0, color="white", linestyle="--", linewidth=1.0)

    if initial_points is None and x_initial is not None and y_initial is not None:
        initial_points = np.column_stack((np.ravel(x_initial), np.ravel(y_initial)))
    initial_points = _as_2d_points(initial_points)
    exploration_points = _as_2d_points(exploration_points)
    trace_parts = [points for points in (initial_points, exploration_points) if points is not None]
    if trace_parts:
        trace_points = np.vstack(trace_parts)
        ax.plot(
            trace_points[:, 0],
            trace_points[:, 1],
            color="black",
            linestyle=":",
            linewidth=0.8,
            label="search trajectory",
            zorder=4,
        )
    if initial_points is not None:
        ax.scatter(
            initial_points[:, 0],
            initial_points[:, 1],
            color="red",
            s=35,
            marker="o",
            label="initial points",
            zorder=5,
        )
    if exploration_points is not None:
        ax.scatter(
            exploration_points[:, 0],
            exploration_points[:, 1],
            color="black",
            s=35,
            marker="o",
            label="exploration points",
            zorder=5,
        )
    plt.tight_layout()

    return fig, ax


# Preview the scalar robust weighted objective used by the optimizer.
_preview_weights = (0.5, 0.5)
_preview_seed = 12345

def preview_robust_weighted_average(X):
    return robust_weighted_average(
        X,
        weights=_preview_weights,
        n_delta=512,
        delta_std=0.04,
        rng=np.random.default_rng(_preview_seed),
        clip=True,
    )

fig, ax = plot_synthetic_function_3d(preview_robust_weighted_average)
ax.set_title("robust weighted average: 0.5 f1 + 0.5 f2")
plt.show()


In [ ]:
# -------------------- robust LCB on robust weighted-average MO target --------------------
# This notebook keeps the robust LCB-on-J loop from gaussian_process_debug_robust.ipynb.
# The black-box scalar target is robust_weighted_average from the multi-objective function above.

# -------------------- 1. Synthetic function switch --------------------
objective_weights = (0.5, 0.5)
objective_n_delta = 512
objective_delta_std = 0.04
objective_seed = 12345


def f_true(X):
    # Reinitialize the RNG on every call so the Monte Carlo scalarization is deterministic.
    # This keeps the GP noise-free unless noise_std below is changed explicitly.
    return robust_weighted_average(
        X,
        weights=objective_weights,
        func=robust_pareto_ridge,
        n_delta=objective_n_delta,
        delta_std=objective_delta_std,
        rng=np.random.default_rng(objective_seed),
        clip=True,
    )


function_name = "robust_weighted_average_robust_pareto_ridge"
bounds = np.array([[0.0, 1.0], [0.0, 1.0]])
d = 2
gamma = 40.0

Sigma = np.diag((0.05 * (bounds[:, 1] - bounds[:, 0])) ** 2)
canonical_name = function_name
lower_bounds = bounds[:, 0]
upper_bounds = bounds[:, 1]

# -------------------- 2. Default experiment settings --------------------
random_seed = 0
rng = np.random.default_rng(random_seed)
noise_std = 0.0
noise_var = noise_std ** 2
n_initial = 5 * d
n_iter = 30
n_perturb_acq = 64
n_perturb_validation = 512
kappa = 1.0
n_candidates = 2000 if d <= 2 else 5000

STRATEGY = "robust_lcb_J"
print(f"--- Strategy Selected: {STRATEGY} ---")
print(f"--- Function Selected: {canonical_name} (d={d}) ---")
print(f"--- Objective weights: {objective_weights} ---")
print(f"--- Objective perturbation std: {objective_delta_std} ---")

# -------------------- 3. Initialization --------------------
X_train = rng.uniform(lower_bounds, upper_bounds, size=(n_initial, d))
y_train = f_true(X_train).reshape(-1, 1)
if noise_std > 0:
    y_train = y_train + rng.normal(0.0, noise_std, size=y_train.shape)
X_initial = X_train.copy()
y_initial = y_train.copy()
exploration_points = []

# -------------------- 4. Bayesian Optimization Loop --------------------
print(f"--- Starting Optimization Loop ({n_iter} iterations) ---")
print(f"{'Iter':<5} | {'Best y':<12} | {'New y':<12} | {'acq value':<14}")
print("-" * 60)

x_robust_lcb_J = None
robust_lcb_history = []
for i in range(n_iter):
    # 1. Fit GP to the current robust weighted-average observations D={(X, S_w)}.
    gp = lib_debug.KernelGPRegressor(
        kernel=lib_debug.rbf_kernel,
        gamma=gamma,
        noise_var=noise_var,
    ).fit(X_train, y_train)

    # 2. Minimize robust LCB on J(x)=E_delta[S_w(x+delta)] by random search.
    x_next, acq_value = lib_debug.optimize_robust_lcb_by_random_search(
        gp,
        Sigma,
        bounds,
        rng,
        n_perturb=n_perturb_acq,
        kappa=kappa,
        n_candidates=n_candidates,
    )

    # 3. Observe the scalar robust weighted-average target once at x_next.
    y_next = f_true(x_next.reshape(1, -1)).reshape(1, 1)
    if noise_std > 0:
        y_next = y_next + rng.normal(0.0, noise_std, size=(1, 1))

    # 4. Update data.
    X_train = np.vstack([X_train, x_next])
    y_train = np.vstack([y_train, y_next])
    x_robust_lcb_J = x_next
    exploration_points.append(x_next.copy())
    robust_lcb_history.append(acq_value)

    print(
        f"{i + 1:<5} | {np.min(y_train):<12.6f} | "
        f"{y_next.item():<12.6f} | {acq_value:<14.6f}"
    )

# -------------------- 5. Final surrogate robust mean validation --------------------
gp = lib_debug.KernelGPRegressor(
    kernel=lib_debug.rbf_kernel,
    gamma=gamma,
    noise_var=noise_var,
).fit(X_train, y_train)

best_idx_final = int(np.argmin(y_train))
best_observed_x = X_train[best_idx_final]
best_observed_y = float(y_train[best_idx_final, 0])

validation = lib_debug.validate_surrogate_robust_mean(
    {
        "robust_lcb_J": x_robust_lcb_J,
        "best_observed": best_observed_x,
    },
    gp,
    Sigma,
    bounds,
    n_mc=n_perturb_validation,
    rng=rng,
    use_cov=True,
)
robust_best = min(validation, key=lambda name: validation[name]["posterior_robust_mean"])

# Dense-grid check of the true scalar target for a simple reference optimum.
grid_n = 121
grid_x1 = np.linspace(0.0, 1.0, grid_n)
grid_x2 = np.linspace(0.0, 1.0, grid_n)
Grid_X1, Grid_X2 = np.meshgrid(grid_x1, grid_x2)
Grid_X = np.stack([Grid_X1.ravel(), Grid_X2.ravel()], axis=-1)
Grid_Y = f_true(Grid_X)
grid_best_idx = int(np.argmin(Grid_Y))
grid_best_x = Grid_X[grid_best_idx]
grid_best_y = float(Grid_Y[grid_best_idx])

# -------------------- 6. Required summaries --------------------
print("\nBO summary")
print(f"function name: {canonical_name}")
print(f"dimension d: {d}")
print(f"bounds:\n{bounds}")
print(f"objective weights: {objective_weights}")
print(f"objective_n_delta: {objective_n_delta}")
print(f"objective_delta_std: {objective_delta_std}")
print(f"n_initial: {n_initial}")
print(f"n_iter: {n_iter}")
print(f"n_perturb_acq: {n_perturb_acq}")
print(f"kappa: {kappa}")
print(f"Sigma:\n{Sigma}")
print(f"best observed y: {best_observed_y:.6f}")
print(f"best observed x: {best_observed_x}")
print(f"grid best y: {grid_best_y:.6f}")
print(f"grid best x: {grid_best_x}")
print(f"final robust_lcb_J recommended x: {x_robust_lcb_J}")

print("\nvalidation summary")
header = (
    f"{'candidate':<18} {'nominal_posterior_mean':>24} "
    f"{'nominal_posterior_std':>23} {'posterior_robust_mean':>24} "
    f"{'posterior_robust_std':>23} {'posterior_mu_sample_std':>25}"
)
print(header)
print("-" * len(header))
for candidate, stats in validation.items():
    print(
        f"{candidate:<18} {stats['nominal_posterior_mean']:>24.6f} "
        f"{stats['nominal_posterior_std']:>23.6f} "
        f"{stats['posterior_robust_mean']:>24.6f} "
        f"{stats['posterior_robust_std']:>23.6f} "
        f"{stats['posterior_mu_sample_std']:>25.6f}"
    )
print(f"\nrobust best by posterior_robust_mean: {robust_best}")

# -------------------- 7. Overplot final robust LCB J recommendation --------------------
final_robust_lcb_J_recommendation = {
    "function_name": canonical_name,
    "x": np.asarray(x_robust_lcb_J, dtype=float).copy(),
    "y": float(f_true(np.asarray(x_robust_lcb_J, dtype=float).reshape(1, -1))[0]),
}
print("\nfinal robust LCB J recommendation")
print(f"function name: {final_robust_lcb_J_recommendation['function_name']}")
print(f"x: {final_robust_lcb_J_recommendation['x']}")
print(f"y: {final_robust_lcb_J_recommendation['y']:.6f}")

fig, ax = plot_synthetic_function_3d(
    f_true,
    initial_points=X_initial,
    exploration_points=np.asarray(exploration_points),
)
ax.scatter(
    [grid_best_x[0]],
    [grid_best_x[1]],
    color="cyan",
    edgecolor="black",
    s=90,
    marker="^",
    label="grid best robust weighted average",
    zorder=6,
)
ax.scatter(
    [x_robust_lcb_J[0]],
    [x_robust_lcb_J[1]],
    color="magenta",
    edgecolor="black",
    s=90,
    marker="*",
    label="final robust LCB J recommendation",
    zorder=7,
)
ax.legend(loc="upper left")
plt.show()

# -------------------- 8. robust_LCB tracking plot --------------------
epochs = np.arange(1, len(robust_lcb_history) + 1)
fig, ax = plt.subplots()
ax.plot(epochs, robust_lcb_history, marker="o")
ax.set_xlabel("epoch")
ax.set_ylabel("robust_LCB")
plt.show()
